# ClaimFlow imaging models, Colab training

Trains the two stage-1 imaging models on a GPU runtime:

1. **Modality classifier** (ct / mri / xray) on ROCOv2 radiographs (streamed, CUI-mapped).
2. **Authenticity detector** (fake / real) on a tampering-derived set built from the same images.

Flow: parameters -> installs -> get the code -> stage or build the dataset -> build the
authenticity set -> train both models -> evaluate on the test split -> download the weights.

**Runtime > Change runtime type > GPU (T4 is plenty).**

In [ ]:
# ---- parameters (edit here only) ----
PER_CLASS = 5000          # images per modality class
SIZE = 512                # long edge of saved images
EPOCHS_MODALITY = 15
EPOCHS_AUTH = 12
BATCH = 32
SEED = 42
INPUT_SIZE = 224

# Dataset staging: set True if the built imaging set already lives on your Drive
# (skips the ~1-2h streaming build on later runs).
USE_DRIVE_DATASET = False
DRIVE_DATA = "/content/drive/MyDrive/claimflow/imaging"

## Installs

Colab ships a CUDA build of torch + torchvision already, so we only add the rest.
(If you ever need to pin torch, use the CUDA wheel index, not the CPU one.)

In [ ]:
%pip install -q timm datasets pillow opencv-python-headless numpy
import torch
print("torch", torch.__version__, "cuda:", torch.cuda.is_available())

## Get the code (the repo is private, so no `git clone`)

On your machine, zip the training package from `backend/` and upload it below:

```bash
cd backend
zip -r ml_training.zip ml_training -x "ml_training/data/imaging/*" -x "ml_training/data/authenticity/*" -x "*__pycache__*"
```

The next cell prompts for the zip (skipped if `ml_training/` is already present, e.g.
after a runtime restart). Alternative: keep the zip on Drive and copy it in.

In [ ]:
import os
import zipfile
from pathlib import Path

ROOT = Path("/content/claimflow")
ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)

if not (ROOT / "ml_training" / "datasets" / "build_datasets.py").exists():
    from google.colab import files
    uploaded = files.upload()  # choose ml_training.zip
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        z.extractall(ROOT)

assert (ROOT / "ml_training" / "datasets" / "build_datasets.py").exists(), "upload failed"
print("code ready at", ROOT)

## Stage the dataset from Drive (optional)

If `USE_DRIVE_DATASET = True`, the imaging set staged at
`/content/drive/MyDrive/claimflow/imaging` (with its `manifest.csv`) is symlinked in
and the streaming build below is skipped. Tip: after the first successful build, copy
`ml_training/data/imaging` to Drive so future sessions start here.

In [ ]:
data_dir = ROOT / "ml_training" / "data"
data_dir.mkdir(parents=True, exist_ok=True)
imaging_dir = data_dir / "imaging"

if USE_DRIVE_DATASET:
    from google.colab import drive
    drive.mount("/content/drive")
    if not imaging_dir.exists():
        imaging_dir.symlink_to(DRIVE_DATA)
    assert (imaging_dir / "manifest.csv").exists(), f"no manifest under {DRIVE_DATA}"
    print("using Drive-staged dataset:", imaging_dir.resolve())
else:
    print("will build the dataset by streaming ROCOv2 (next cell)")

## Build the imaging set (skipped if already staged)

Streams `eltorio/ROCOv2-radiology`, maps rows to ct/mri/xray via the validated CUI
sets, and saves grayscale JPEGs until every class hits `PER_CLASS`. Resumable: rerun
the cell after an interruption and it continues where it stopped.

In [ ]:
if (imaging_dir / "manifest.csv").exists() and USE_DRIVE_DATASET:
    print("dataset already staged, skipping build")
else:
    !python -m ml_training.datasets.build_datasets --source rocov2 \
        --per-class {PER_CLASS} --size {SIZE} --out ml_training/data/imaging

## Build the authenticity set

One REAL + one FAKE sample per source image; both classes get the identical final
JPEG re-save (class-shortcut killer) and the split is keyed on the source image id,
so a source never straddles train/val/test.

In [ ]:
!python -m ml_training.datasets.build_datasets --authenticity \
    --real-dir ml_training/data/imaging --out ml_training/data/authenticity --seed {SEED}

## Train the modality classifier (ct / mri / xray)

ImageNet-pretrained EfficientNet-B0, head warmup (2 frozen epochs), cosine schedule,
early stop on val macro-F1. Train augs include JPEG-quality jitter + blur so the model
cannot shortcut on per-source compression signatures.

In [ ]:
!python -m ml_training.train_modality --data-dir ml_training/data/imaging \
    --epochs {EPOCHS_MODALITY} --batch-size {BATCH} --input-size {INPUT_SIZE} \
    --seed {SEED} --device auto --out weights/

## Train the authenticity detector (fake / real)

Same recipe but GEOMETRIC-ONLY augmentation: JPEG/blur augs would erase the forensic
artifacts (double-JPEG ghosts, splice seams, resampling softness) the model must learn.

In [ ]:
!python -m ml_training.train_authenticity --data-dir ml_training/data/authenticity \
    --epochs {EPOCHS_AUTH} --batch-size {BATCH} --input-size {INPUT_SIZE} \
    --seed {SEED} --device auto --out weights/

## Evaluate on the held-out test split

Accuracy, macro-F1, per-class precision/recall, confusion matrices, and ECE
before/after temperature scaling. Writes `ml_training/data/eval_report.json`.

In [ ]:
!python -m ml_training.evaluate --weights-dir weights/ --data-dir ml_training/data \
    --report ml_training/data/eval_report.json

## Download the trained weights

Zips `weights/*.pt` + `*_config.json` (+ the eval report). Unzip into `backend/weights/`
locally; the serving RealAnalyzer reads the config JSONs for classes, input size,
normalization, and temperature.

In [ ]:
import shutil

from google.colab import files

shutil.copy("ml_training/data/eval_report.json", "weights/eval_report.json")
archive = shutil.make_archive("claimflow_weights", "zip", "weights")
print("created", archive)
files.download(archive)